In [ ]:
# | default_exp tokenisation.sentencepiece

# SentencePiece


In [ ]:
import lorem

from ai_notes.tokenisation.bpe import BPE, NaiveBPE

In [ ]:
naive_bpe = NaiveBPE(vocab_size=300)
naive_bpe.train(lorem.text())
encoded_text = naive_bpe.encode("hello world")

In [ ]:
print(encoded_text)

[104, 101, 108, 108, 111, 32, 119, 258, 108, 100]


## Dummy whitespace


In our [byte_pair_encoding_from_scratch.ipynb](./byte_pair_encoding_from_scratch.ipynb) we used a regex to split text into chunks. Merges can can never take place across those hard world boundaries.

SentencePiece runs on the entire input as a single character stream and lets merges happen freely across any adjacent characters.

It replaces every space character with the special `\u2581` (▁) character (which resembles an underscore but is much rarer so it won't collide with real underscores).

Because ▁ can merge with any other character, word boundary information is carried inside the token itself. We end up with merges like `▁` + `h` -> `▁he`.

We prepend ▁ to the start of an input as well. This helps the model avoid encoding a word differently just because it happened to appear at the start of the input.

It's pretty easy to implement this in python. We can take our NaiveBPE implementation and just tweak the `get_tokens` and `decode` methods.

We add these two lines to the start of the `get_tokens` method:

```python
text = f" {text}"
text = text.replace(" ", self.space_token)
```

And this line to the end of the `decode` method:

```python
text = text.replace(self.space_token, " ").strip()
```


In [ ]:
# | export
class SentencePieceBPE:
    def __init__(self, vocab_size: int):
        self.vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.merges: dict[tuple[int, int], int] = {}
        self.vocab_size = vocab_size
        self.space_token = "\u2581"

    def encode(self, text: str) -> list[int]:
        tokens = self.get_tokens(text)
        for pair, idx in self.merges.items():
            tokens = self.merge_pair(tokens, pair, idx)
        return tokens

    def decode(self, tokens: list[int]) -> str:
        token_bytes: bytes = b"".join(self.vocab[token] for token in tokens)
        text = token_bytes.decode("utf-8", errors="replace")
        text = text.replace(self.space_token, " ").strip()
        return text

    def train(self, text: str):
        tokens = self.get_tokens(text)
        for i in range(256, self.vocab_size):
            pair_counts = self.get_pair_counts(tokens)
            most_common_pair = max(pair_counts, key=lambda p: pair_counts[p])
            tokens = self.merge_pair(tokens, most_common_pair, i)
            self.merges[most_common_pair] = i
            self.vocab[i] = self.vocab[most_common_pair[0]] + self.vocab[most_common_pair[1]]

    def get_tokens(self, text: str) -> list[int]:
        text = f" {text}"
        text = text.replace(" ", self.space_token)
        tokens: bytes = text.encode(encoding="utf-8")
        return list(map(int, tokens))

    def get_pair_counts(self, tokens: list[int]) -> dict[tuple[int, int], int]:
        counts: dict[tuple[int, int], int] = {}
        for pair in zip(tokens, tokens[1:]):
            counts[pair] = counts.get(pair, 0) + 1
        return counts

    def merge_pair(self, tokens: list[int], pair: tuple[int, int], idx: int) -> list[int]:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(idx)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        return new_tokens

In [ ]:
sentencepiece_bpe = SentencePieceBPE(vocab_size=300)
sentencepiece_bpe.train(lorem.text())
encoded_text = sentencepiece_bpe.encode("hello world")

In [ ]:
encoded_text

[257, 104, 101, 108, 108, 111, 257, 119, 259, 108, 100]

In [ ]:
decoded_text = sentencepiece_bpe.decode(encoded_text)
decoded_text

'hello world'

## Byte fallback
